# `d3rlpy` tutorial

In [30]:
import d3rlpy
from d3rlpy.datasets import get_cartpole
from d3rlpy.algos import DQNConfig, TD3Config
from d3rlpy.metrics import TDErrorEvaluator, EnvironmentEvaluator
import gymnasium as gym
from pmind.replay import convert_rb_to_dataset

import numpy as np

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Getting started

In [31]:
# Prepare dataset
# Dataset = replay buffer. We can use integrated dataset or create our own!
dataset, env = get_cartpole()

# Setup algorithm
dqn = DQNConfig().create(device=None)
# NOTE: we can put most of cfg here!

# Initialize NN with right obs and action dims
dqn.build_with_dataset(dataset) 

# Setup metrics

# This metric suggests how Q functions overfit to training sets. 
# If the TD error is large, the Q functions are overfitting.
td_error_evaluator = TDErrorEvaluator(episodes=dataset.episodes)

env_evaluator = EnvironmentEvaluator(env)

rewards = env_evaluator(dqn, dataset=None)
print(f"Reward at initialization: {rewards}")

2026-04-29 20:41.20 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2026-04-29 20:41.20 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2026-04-29 20:41.20 [info     ] Action size has been automatically determined. action_size=2
Reward at initialization: 70.6


/Users/vlad/Documents/University/Master-MIND/projet-mind/.venv/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [57]:
dataset.sample_transition_batch(10)

TransitionMiniBatch(observations=array([[ 4.07844074e-02, -9.35439229e-01, -1.85595050e-01,
         3.62726212e-01],
       [-2.55410582e-01, -5.99204481e-01,  1.67082101e-01,
         6.41042829e-01],
       [ 5.17938554e-01,  3.18980515e-01,  1.01432122e-01,
         6.44903839e-01],
       [ 1.18488193e-01,  1.72797054e-01, -1.64146349e-01,
        -4.45323616e-01],
       [-2.07064748e+00, -5.33547163e-01,  1.62998259e-01,
         4.30697352e-01],
       [-2.17780089e+00, -2.16442370e+00, -1.15185775e-01,
        -4.39078718e-01],
       [-1.48486257e-01,  4.16570812e-01,  1.67225540e-01,
        -1.65996745e-01],
       [-5.27374625e-01, -5.43839455e-01,  4.91389334e-02,
         6.43623471e-01],
       [ 3.18207801e-03,  1.45973593e-01,  1.47213798e-03,
        -2.89272875e-01],
       [-1.60615861e+00,  7.62329102e-01, -7.84145147e-02,
        -1.40828311e-01]], dtype=float32), actions=array([[0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
 

In [ ]:
dataset.sample_transition_batch(10)

TransitionMiniBatch(observations=array([[-4.1671332e-02,  3.9147311e-01, -1.9471956e-02, -5.8830458e-01],
       [ 2.8097913e-01,  6.9776821e-01, -1.0430891e-01, -3.7472779e-01],
       [-4.0468123e-02,  8.8299388e-01,  1.4337866e-01, -5.1602817e-01],
       [-6.3360557e-02,  5.1560777e-01,  1.8194871e-01,  1.8011546e-01],
       [-3.6844634e-02,  5.6884456e-01,  1.9230783e-01,  1.5330592e-01],
       [ 1.0370362e+00,  5.1159924e-03, -7.6266117e-02,  8.2269602e-02],
       [ 4.2834505e-01,  1.2926066e+00,  1.0033613e-01, -2.0966490e-01],
       [ 9.9408530e-02,  1.8143517e-01, -1.5431182e-01, -4.7236389e-01],
       [-7.0834655e-01, -1.8494633e+00, -2.3224160e-04,  1.0096819e+00],
       [ 1.5716295e-01, -3.4429675e-01, -1.4919947e-01,  1.0414939e-01]],
      dtype=float32), actions=array([[0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.]], dtype=float32), rewards=array([[1.],
       [1.],
       [1.],
       [1.],

In [60]:
from copy import deepcopy
dataset_test = deepcopy(dataset)
for episode in dataset.episodes:
    dataset_test.append_episode(episode)

In [93]:
episode.terminated

False

In [66]:
test = dataset.sample_transition_batch(10)

In [ ]:
import numpy as np
from d3rlpy.dataset import MDPDataset


def get_from_dataset(dataset, var_name):
    var = []
    for episode in dataset.episodes:
        if var_name == "terminals":
            terminals = np.zeros(len(episode))
            if episode.terminated:
                terminals[-1] = 1
            var.extend(terminals)
        elif var_name == "timeouts":
            timeouts = np.zeros(len(episode))
            if not episode.terminated:
                timeouts[-1] = 1
            var.extend(timeouts)
        else:
            var.extend(getattr(episode, var_name))
    return np.array(var)


def mix_datasets_transitions(dataset_a, dataset_b, n_samples_a, n_samples_b):

    obs_a = get_from_dataset(dataset_a, "observations")
    act_a = get_from_dataset(dataset_a, "actions")
    rew_a = get_from_dataset(dataset_a, "rewards")
    term_a = get_from_dataset(dataset_a, "terminals")
    time_a = get_from_dataset(dataset_a, "timeouts")

    obs_b = get_from_dataset(dataset_b, "observations")
    act_b = get_from_dataset(dataset_b, "actions")
    rew_b = get_from_dataset(dataset_b, "rewards")
    term_b = get_from_dataset(dataset_b, "terminals")
    time_b = get_from_dataset(dataset_b, "timeouts")

    idx_a = np.random.choice(len(obs_a), n_samples_a, replace=False)
    idx_b = np.random.choice(len(obs_b), n_samples_b, replace=False)

    observations = np.concatenate([obs_a[idx_a], obs_b[idx_b]])
    actions = np.concatenate([act_a[idx_a], act_b[idx_b]])
    rewards = np.concatenate([rew_a[idx_a], rew_b[idx_b]])
    terminals = np.concatenate([term_a[idx_a], term_b[idx_b]])
    timeouts = np.concatenate([time_a[idx_a], time_b[idx_b]])

    # TODO: need to mix them or they get shuffled anyway? hope so...

    return MDPDataset(observations, actions, rewards, terminals, timeouts)

In [106]:
mix_datasets_transitions(dataset, dataset_test, 10, 10)

2026-04-29 21:25.45 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2026-04-29 21:25.45 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2026-04-29 21:25.45 [info     ] Action size has been automatically determined. action_size=2


In [ ]:
# Offline training
dqn.fit(
    dataset,
    n_steps=10000,
    evaluators={
        'td_error': td_error_evaluator,
        'environment': env_evaluator
    }
    )

AttributeError: 'list' object has no attribute 'dataset_info'

In [14]:
# Use trained policy
observation, _ = env.reset()
action = dqn.predict(observation.reshape(1,-1)) # don't forget to add batch dimension
value = dqn.predict_value(observation.reshape(1,-1), action)

In [15]:
# Save and load
if False:
    # save full parameters and configurations in a single file.
    dqn.save('dqn.d3')
    # load full parameters and build algorithm
    dqn2 = d3rlpy.load_learnable("dqn.d3")

    # save full parameters only
    dqn.save_model('dqn.pt')
    # load full parameters with manual setup
    dqn3 = DQNConfig().create(device=None)
    dqn3.build_with_dataset(dataset)
    dqn3.load_model('dqn.pt')

    # save the greedy-policy as TorchScript
    dqn.save_policy('policy.pt')
    # save the greedy-policy as ONNX
    # dqn.save_policy('policy.onnx')

# Data collection

Not really relevant for us, since it doesn't provide uniform exploration and we better convert our replay buffers to their format

# Create our dataset

Note that whereas valid trajectories are pretty straightforward to convert to `d3rlpy` replay buffer, the uniform sampling results in "jumps" and needs a special treatment - we should organize them as episodes of sie 1 I guess.

In [16]:
# vector observation
# 1000 steps of observations with shape of (100,)
observations = np.random.random((1000, 100))

# 1000 steps of actions with shape of (4,)
actions = np.random.random((1000, 4))

# 1000 steps of rewards
rewards = np.random.random(1000)

# 1000 steps of terminal flags
# terminals = np.random.randint(2, size=1000)
terminals = np.zeros(1000)
timeouts = np.random.randint(2, size=1000)

dataset = d3rlpy.dataset.MDPDataset(
    observations=observations,
    actions=actions,
    rewards=rewards,
    terminals=terminals,
    timeouts=timeouts,
)

2026-04-27 17:09.14 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float64')], shape=[(4,)]) observation_signature=Signature(dtype=[dtype('float64')], shape=[(100,)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2026-04-27 17:09.14 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2026-04-27 17:09.14 [info     ] Action size has been automatically determined. action_size=4


# Pre / post processing

We are interested only for action pre/post processing for `Pendulum` to normalize action to `[-1.0, 1.0]` and then denormalize it back. 

Do we need to scale observations (we didn't do it in BBRL)?

We won't preprocess rewards neither.

In [17]:
action_scaler = d3rlpy.preprocessing.MinMaxActionScaler(
    minimum=np.array(-2.0),
    maximum=np.array(2.0),
)
# NOTE: we can deduce min max from cfg.action_scaling

sac = d3rlpy.algos.SACConfig(action_scaler=action_scaler).create()

# Customize NN

You can define a custom NN with `torch.nn.Module` or use their `VectorEncoderFactory` for an MLP.

In [18]:
# Can do encoder factory
encoder_factory = d3rlpy.models.VectorEncoderFactory(
    hidden_units=[300, 400],
    activation="tanh",
)
# NOTE: we can put cfg.algorithm.architecture.[actor/critic]_hidden_size
td3 = d3rlpy.algos.TD3Config(
    actor_encoder_factory=encoder_factory, critic_encoder_factory=encoder_factory
).create()


# Online RL & Fine-tuning

No interest.

# Use trained policies

In [19]:
# # save d3 file
# cql_old.save("model.d3")
# # reconstruct full setup from a d3 file
# cql = d3rlpy.load_learnable("model.d3")


# # Option 2: Load pt file

# # save pt file
# cql_old.save_model("model.pt")
# # setup algorithm manually
# cql = d3rlpy.algos.CQLConfig().create()

# # choose one of three to build PyTorch models

# # if you have MDPDataset object
# cql.build_with_dataset(dataset)
# # or if you have Gym-styled environment object
# cql.build_with_env(env)
# # or manually set observation shape and action size
# cql.create_impl((3,), 1)

# # load pretrained model
# cql.load_model("model.pt")

For inference, don't forget the batch dimension.

In [ ]:
dqn.predict(env.observation_space.sample().reshape(1,-1))
# TODO: but it uses the last policy and not the best one?
# TODO: make a `model` object with the same interface that we used for BBRL

array([0])